In [1]:
# Path + packaging
import sys  # no installation needed
from pathlib import Path  # no installation needed

SRC_DIR = Path(r"C:\Users\quantbase\Desktop\SyStrat\src")
PKG_DIR = SRC_DIR / "syslib"
PKG_DIR.mkdir(parents=True, exist_ok=True)
(PKG_DIR / "__init__.py").touch(exist_ok=True)  # ensure it's importable

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("sys.path OK:", SRC_DIR in map(Path, map(str, sys.path)))


sys.path OK: True


In [2]:
from pathlib import Path
import importlib
import datetime as date
from datetime import date
import syslib.wp_core as wp_core


RUN_DATE = date.today().strftime("%d-%m-%Y") #"30-10-2025"
BASE = Path(r"C:\Users\quantbase\Desktop\SyStrat") / RUN_DATE
OUT = BASE / "figures"

In [3]:
symbols = ['BTC-USD','ETH-USD','BNB-USD','XRP-USD','ADA-USD','LINK-USD','SOL-USD']
weights   = [0.23, 0.078, 0.078, 0.078, 0.078, 0.078, 0.078]

In [4]:
import syslib.size_legacy as sl
import syslib.size_ml as sml

In [5]:
# project-local helpers
import syslib.size_legacy as sl      # project-local
import syslib.size_ml as sml         # project-local
import tools.portfolio_summary as ps # project-local
importlib.reload(sl); importlib.reload(sml); importlib.reload(ps)

weights = {
    "BTC-USD":0.23, "ETH-USD":0.078, "BNB-USD":0.078,
    "XRP-USD":0.078, "ADA-USD":0.078, "LINK-USD":0.078, "SOL-USD":0.078
}
mult = {t: 1.0 for t in weights}  # set futures/contract multipliers here if any

In [21]:
# --- sizing knobs (match what you used in runs) ---
cfg = sl.SizeConfig(
    notional=50_000.0, leverage=2.0, risk_fraction=0.02/19,
    window_len=32, w0=0.6, decay=0.4,
    hist_span=54, blend_recent=0.7, scale=19.0,
    lookback_years=5, interval="1d",
)

In [20]:
# --- EWMA snapshot ---
snap_ewma = sl.size_snapshot_ewma(weights, mult_map=mult, cfg=cfg, prices=None)

In [22]:
# --- ML snapshot: prefer per-ticker dir; else join preds_test with rowmap ---
pdir   = BASE / "data_int/ml/preds_test_k10"
rowmap = BASE / "data_int/ml/rowmap_test_k10.parquet"

In [23]:
provider_kwargs = {"base_dir": BASE, "k_label": 10, "model_tag": "xgb"}
if pdir.exists():
    provider_kwargs["per_ticker_dir"] = pdir
else:
    provider_kwargs["x_test_path"]   = rowmap
    provider_kwargs["x_date_col"]    = "date"
    provider_kwargs["x_ticker_col"]  = "ticker"

In [24]:
snap_ml = sml.size_snapshot_ml(weights, provider_kwargs, mult_map=mult, cfg=cfg, prices=None)

In [25]:
display(snap_ewma)
display(snap_ml)


,portw,mult,last_price,pred_vol,units,notional_alloc
ticker,,,,,,
ADA-USD,0.078,1.0,0.396584,0.351869,1117.913836,443.346746
BNB-USD,0.078,1.0,877.806885,0.266414,0.667066,585.555508
BTC-USD,0.230,1.0,88175.179688,0.193480,0.026963,2377.505516
ETH-USD,0.078,1.0,3060.594727,0.334396,0.152426,466.512770
LINK-USD,0.078,1.0,13.292393,0.341377,34.378523,456.972826
SOL-USD,0.078,1.0,129.481094,0.297678,4.047361,524.056718
XRP-USD,0.078,1.0,1.979287,0.358497,219.851825,435.149866


,portw,mult,last_price,pred_vol,units,notional_alloc
ticker,,,,,,
ADA-USD,0.078,1.0,0.396584,0.040196,515.059143,204.264217
BNB-USD,0.078,1.0,877.806885,0.030646,0.305205,267.910976
BTC-USD,0.230,1.0,88175.179688,0.024638,0.011144,982.668198
ETH-USD,0.078,1.0,3060.594727,0.033025,0.081232,248.617878
LINK-USD,0.078,1.0,13.292393,0.040766,15.152011,201.406486
SOL-USD,0.078,1.0,129.481094,0.040658,1.559621,201.941432
XRP-USD,0.078,1.0,1.979287,0.034719,119.481008,236.487210


In [13]:
#-------Portfolio Summary

In [26]:
# optional: regime labels per ticker if you have them
regime_map = {

}

summary = ps.make_portfolio_summary(
    weights=weights,
    snap_ewma=snap_ewma,
    snap_ml=snap_ml,
    regime_map=regime_map,
    ml_ann_shim=19.0,   # your quick annualization shim
)
display(summary)


,portw,last_price,mult,pred_vol_ml,pred_vol_ml_shim,units_ml,notional_alloc_ml,pred_vol_ewma,units_ewma,notional_alloc_ewma,regime,units_diff,vol_ratio_ml_to_ewma
BTC-USD,0.230,88175.179688,1.0,0.024638,0.468113,0.011144,982.668198,0.193480,0.026963,2377.505516,n/a,-0.015819,0.127339
ADA-USD,0.078,0.396584,1.0,0.040196,0.763717,515.059143,204.264217,0.351869,1117.913836,443.346746,n/a,-602.854694,0.114235
BNB-USD,0.078,877.806885,1.0,0.030646,0.582283,0.305205,267.910976,0.266414,0.667066,585.555508,n/a,-0.361862,0.115033
ETH-USD,0.078,3060.594727,1.0,0.033025,0.627469,0.081232,248.617878,0.334396,0.152426,466.512770,n/a,-0.071194,0.098759
LINK-USD,0.078,13.292393,1.0,0.040766,0.774553,15.152011,201.406486,0.341377,34.378523,456.972826,n/a,-19.226511,0.119416
SOL-USD,0.078,129.481094,1.0,0.040658,0.772501,1.559621,201.941432,0.297678,4.047361,524.056718,n/a,-2.487740,0.136584
XRP-USD,0.078,1.979287,1.0,0.034719,0.659655,119.481008,236.487210,0.358497,219.851825,435.149866,n/a,-100.370817,0.096845
__TOTAL__,0.698,NaN,NaN,NaN,NaN,NaN,2343.296396,NaN,NaN,5289.099949,NaN,NaN,NaN


In [27]:
paths = ps.save_portfolio_summary(summary, OUT, stem="portfolio_summary_k10")
print(paths)  # -> {'csv': '...', 'json': '...'}


{'csv': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\figures\\portfolio_summary_k10.csv', 'json': 'C:\\Users\\quantbase\\Desktop\\SyStrat\\17-12-2025\\figures\\portfolio_summary_k10.json'}
